# From Model to Production: Agentic AI with Kiro on SageMaker (AIM306)

Take an open-weight LLM (**GPT-OSS-20B**) to a deployed, benchmarked Amazon SageMaker AI
endpoint — **without writing the deployment code by hand**.

On stage this is two prompts to **Kiro** ("deploy GPT-OSS-20B", then "benchmark it").
Kiro fires the matching `SKILL.md` contract and runs the reference scripts below. This
notebook is the same flow you can step through by hand.

> **Agents are non-deterministic. Skills make them definitive.**

## 0. Confirm the environment

Nothing is hardcoded — region, account, execution role, and bucket are resolved from the
live environment. This same notebook runs unchanged in any account.

In [ ]:
import sys, subprocess
sys.path.append("../scripts")
import config
config.summary()  # {region, account_id, execution_role_arn, bucket}

## 1. Deploy (the `sagemaker-deploy` contract)

Dry-run first to see the plan (latest vLLM DLC, instance, tensor-parallel size, S3 weights),
then deploy for real. The deploy takes ~4–8 min and polls to `InService`.

**Capacity tip:** `ml.g6.16xlarge` is the reliable primary. Quota ≠ capacity — keep a
fallback in mind and pre-warm before a live demo.

In [ ]:
# Dry run — prints the deploy plan, creates nothing.
!python ../scripts/deploy.py --model gpt-oss-20b --instance ml.g6.16xlarge

In [ ]:
# Create the endpoint (billable). Note the ENDPOINT_NAME / IC_NAME it prints at the end.
!python ../scripts/deploy.py --model gpt-oss-20b --instance ml.g6.16xlarge --deploy

In [ ]:
# Paste the two handles the deploy printed (the ENDPOINT_NAME / IC_NAME lines):
ENDPOINT = "REPLACE_WITH_ENDPOINT_NAME"
IC = "REPLACE_WITH_IC_NAME"

# Guard: fail loudly HERE if you ran the notebook top-to-bottom without editing the two
# values above — otherwise the placeholder strings flow into every cell below and you'd
# get a cryptic SageMaker "could not find endpoint" error instead of this clear message.
assert ENDPOINT != "REPLACE_WITH_ENDPOINT_NAME", "Edit ENDPOINT above with the deployed endpoint name."
assert IC != "REPLACE_WITH_IC_NAME", "Edit IC above with the deployed inference-component name."
print(f"endpoint = {ENDPOINT}\nic       = {IC}")

## 2. Smoke test

One OpenAI-style chat request. For GPT-OSS-20B (a reasoning model) you'll see a short
`reasoning` channel and the user-facing `content` — exactly why the benchmark uses a
generous output budget.

In [ ]:
!python ../scripts/smoke_test.py --endpoint {ENDPOINT} --ic {IC} \
    --prompt "In one sentence, what is Amazon SageMaker AI?"

## 3. Benchmark (the `sagemaker-benchmark` contract)

Managed **Amazon SageMaker AI inference benchmark** — SageMaker runs **NVIDIA AIPerf** for
us on managed compute and writes the metrics (TTFT, ITL, latency percentiles, throughput)
to S3. No load generator to write or operate.

In [ ]:
# Dry run — prints the workload + benchmark plan.
!python ../scripts/benchmark.py --endpoint {ENDPOINT} --ic {IC}

In [ ]:
# Launch the managed benchmark (billable). Polls to completion; results land in S3.
!python ../scripts/benchmark.py --endpoint {ENDPOINT} --ic {IC} --run

## 4. Observability (CloudWatch)

Prove the load actually hit the endpoint: invocations, model latency, and per-IC
concurrency — all published to CloudWatch for free.

In [ ]:
!python ../scripts/cloudwatch_metrics.py --endpoint {ENDPOINT} --ic {IC} --minutes 30

## 5. Tear down — **endpoints bill while InService**

Run this the moment you're done. Deletes the inference component → endpoint → config → model.

In [ ]:
!python ../scripts/teardown.py --endpoint {ENDPOINT}        # dry run: list what would go
# !python ../scripts/teardown.py --endpoint {ENDPOINT} --yes  # uncomment to actually delete